# Tiny Agent with Tools 🤖

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

## 0) Install dependencies

In [1]:
# TODO ✅ — Install smolagents (with Transformers support) and wikipedia
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.5 MB/s eta 0:00:00


## 1) Define KB

A **knowledge base** is just a list of dictionaries, each with:
- `source`: a short tag used for citation (e.g. `kb:1`)
- `text`: the snippet the agent can retrieve

In [2]:
# TODO ✅ — Define 5–8 knowledge-base snippets.
# Each entry is a dict with 'source' (citation tag) and 'text' (the snippet).
kb_snippets = [
    {
        "source": "kb:1",
        "text": "Agentic AI loops plan, choose tools, and reflect before answering."
    },
    {
        "source": "kb:2",
        "text": "Useful tools for agents include: math, search, and domain-specific lookup."
    },
    {
        "source": "kb:3",
        "text": "Always cite where evidence came from to stay transparent and trustworthy."
    },
    {
        "source": "kb:4",
        "text": "Keep answers concise (2–4 sentences) to respect the reader's time."
    },
    {
        "source": "kb:5",
        "text": "If evidence is missing, say so clearly and propose a follow-up question."
    },
    {
        "source": "kb:6",
        "text": "Tool-calling agents decide at each step which tool to invoke based on the task."
    },
    {
        "source": "kb:7",
        "text": "smolagents is a lightweight Python framework for building tool-calling agents."
    },
]

print("KB entries:", len(kb_snippets))

KB entries: 7


## 2) Define Tools

In **smolagents**, every tool is a subclass of `Tool`. You must define:
- `name` — used by the agent when calling the tool
- `description` — tells the agent *when* to use the tool
- `inputs` — dict mapping parameter names → `{type, description}`
- `output_type` — return type (`"string"`, `"number"`, …)
- `forward(**kwargs)` — the actual logic

In [4]:
from smolagents import Tool, TransformersModel, ToolCallingAgent


# ─────────────────────────────────────────────────────────────
# TODO ✅  KBLookupTool
# ─────────────────────────────────────────────────────────────
class KBLookupTool(Tool):
    """Returns KB snippets whose text contains any keyword from the query."""

    # TODO ✅ — Give the tool a name and description so the agent knows when to use it.
    name = "kb_lookup"
    description = (
        "Look up relevant information from the internal knowledge base. "
        "Pass a short keyword query; returns matching text snippets with citation tags."
    )

    # TODO ✅ — Declare the single input parameter.
    inputs = {
        "query": {
            "type": "string",
            "description": "Keywords to search for in the knowledge base.",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb  # store the list of snippets

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(word in item["text"].lower() for word in q.split())
        ]
        if matches:
            return "\n".join(matches)
        return "No KB match found. Consider proposing a follow-up question."


# ─────────────────────────────────────────────────────────────
# TODO ✅  MathTool
# ─────────────────────────────────────────────────────────────
class MathTool(Tool):
    """Add or multiply two numbers."""

    name = "math_tool"
    description = (
        "Perform basic arithmetic on two numbers. "
        "Supports 'add' (default) and 'multiply') operations."
    )

    # TODO ✅ — Declare inputs: a, b (numbers) and op (operation string).
    inputs = {
        "a": {
            "type": "number",
            "description": "First operand.",
        },
        "b": {
            "type": "number",
            "description": "Second operand.",
        },
        "op": {
            "type": "string",
            "description": "Operation: 'add' or 'multiply'. Defaults to 'add'.",
            "nullable": True,
        },
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            result = a * b
            return f"{a} × {b} = {result}"
        result = a + b
        return f"{a} + {b} = {result}"


# Instantiate both tools
kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

print("Tools ready:", kb_tool.name, "|" , math_tool.name)

Tools ready: kb_lookup | math_tool


## 3) Model (tiny local)

`TransformersModel` wraps any Hugging Face causal-LM.
We use **TinyLlama-1.1B-Chat** — small enough to run on a free Colab CPU/GPU.

> 💡 For even faster testing replace the model ID with `"sshleifer/tiny-gpt2"` (~5 MB).

In [5]:
# TODO ✅ — Instantiate TransformersModel with the model ID and generation params.
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Swap to the line below for ultra-fast CPU testing (model barely generates coherent text):
# MODEL_ID = "sshleifer/tiny-gpt2"

model = TransformersModel(
    model_id=MODEL_ID,
    max_new_tokens=256,   # cap generation length
    temperature=0.1,      # low temperature → more deterministic tool calls
    do_sample=False,      # greedy decoding for reproducibility
)

print("Model ready:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

`ToolCallingAgent` is the smolagents class that:
1. Formats the user query + tool schemas into a prompt
2. Runs the model to decide which tool to call
3. Executes the tool and feeds the result back
4. Repeats until it can produce a final answer

In [6]:
# TODO ✅ — Wire up the agent with both tools, the model, and custom instructions.
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],   # ← register both tools
    model=model,
    max_steps=3,                  # max reasoning steps before giving up
    instructions=(
        "You are a helpful agentic AI. "
        "Use `math_tool` for arithmetic. "
        "Use `kb_lookup` for conceptual or factual questions. "
        "Always include the citation tag (e.g. [kb:1]) when you use KB results. "
        "Keep answers to 2–4 sentences. "
        "If no evidence is found, say so and propose a follow-up question."
    ),
)

print(agent)

## 5) Test queries

We run three queries and print the tool calls + final answers.

In [ ]:
# TODO ✅ — Run all three test queries and inspect tool calls + final answers.
queries = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for query in queries:
    print("\n" + "=" * 60)
    print(f"Q: {query}")
    print("=" * 60)

    answer = agent.run(query)

    # Inspect the tool calls recorded in the last run
    print("\n📞 Tool calls this run:")
    for step in agent.logs:
        if hasattr(step, "tool_name"):
            print(f"  • {step.tool_name}({step.tool_arguments}) → {step.observations}")

    print(f"\n✅ Final answer: {answer}")


Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': {'type': 'number', 'description': 'First operand'}, 'b':        │
│ {'type': 'number', 'description': 'Second operand'}, 'op': {'type': 'string', 'description': "Operation: 'add'  │
│ or 'multiply'. Defaults to 'add'.", 'nullable': True}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': {'type': 'number', 'description': 'First operand'}, 'b':        │
│ {'type': 'number', 'description': 'Second operand'}, 'op': {'type': 'string', 'description': "Operation: 'add'  │
│ or 'multiply'. Defaults to 'add'.", 'nullable': True}}                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Argument a has type 'object' but should be 'number'

[Step 2: Duration 573.81 seconds| Input tokens: 2,770 | Output tokens: 422]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 6) Reflection

**What did you observe?**
- Did the agent correctly route math queries to `math_tool`?
- Did it look up `kb_lookup` for the conceptual question?
- Did it include citation tags like `[kb:1]` in the answer?

**Why might tool calls fail with tiny models?**
Small models like TinyLlama sometimes fail to produce valid JSON tool-call format.
This is expected — production agents use larger instruction-tuned models (GPT-4, Claude, Llama-3-70B).

**Experiment ideas:**
1. Add a `DivisionTool` and test `"Divide 100 by 4."`
2. Extend `kb_snippets` with domain knowledge about your field.
3. Swap the model for a larger one (e.g. `meta-llama/Llama-3.2-3B-Instruct`) and compare reliability.